<a href="https://colab.research.google.com/github/sonalisalvi/careloop-no-show-guard/blob/main/01_ingest_clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CareLoop Clinics — No-Show Guard

## IIT Roorkee Data Science and AI Capstone

**Part 1: Python and Data Foundations**

### Objective
Clean and prepare outpatient appointment data to support
no-show analysis and prediction.

### Tools
- Google Colab
- Python
- CSV
- Pandas
- NumPy
- Open-Meteo API

### Student
Sonali Salvi

# **CareLoop Clinics — No-Show Guard**

## IIT Roorkee Data Science and AI Capstone

### Part 1: Python and Data Foundations

**Business objective:**

Prepare a reliable outpatient appointment dataset
for no-show analysis and prediction.

**Tasks:**

* 1.1 CSV ingestion and validation
* 1.2 Initial data inspection and cleaning
* 1.3 Missing values and outlier treatment
* 1.4 Feature engineering and NumPy analysis
* 1.5 Clinic weather enrichment
* 1.6 Final data export and SQL preparation

In [2]:
# Clone the public IIT project repository

# Exact repository URL created by you
REPO_URL = (
    "https://github.com/sonalisalvi/careloop-no-show-guard.git"
)

# !git clone https://github.com/sonalisalvi/careloop-no-show-guard.git

import subprocess
from pathlib import Path

# Repository Location inside the colab runtime

PROJECT_DIR = Path("/content/careloop-no-show-guard")

# Clone the repository if it is not already present.

if PROJECT_DIR.exists():
   print("Repository folder already exists.")
   print("Updating it from GitHub...")

   subprocess.run(
        ["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"],
        check=True
    )
else:
  print("Cloning the repository...")

  subprocess.run(
        ["git", "clone", REPO_URL, str(PROJECT_DIR)],
        check=True
    )

print("\nRepository connection completed!")
print("Project location:", PROJECT_DIR)

Cloning the repository...

Repository connection completed!
Project location: /content/careloop-no-show-guard


In [3]:
# Define paths to the original dataset

from pathlib import Path

# Main project directory
PROJECT_DIR = Path("/content/careloop-no-show-guard")

# Original dataset folder

DATA_DIR = PROJECT_DIR / "data"

# Individual dataset paths

APPOINTMENTS_PATH = DATA_DIR / "appointments.csv"
CLINICS_PATH = DATA_DIR / "clinics.csv"
FOLLOWUP_NOTES_PATH = DATA_DIR / "followup_notes.csv"

print("Project directory:", PROJECT_DIR)
print("Data directory:", DATA_DIR)

print("\nDataset verification:")

required_files = [
    APPOINTMENTS_PATH,
    CLINICS_PATH,
    FOLLOWUP_NOTES_PATH
]

for file_path in required_files:
    if file_path.exists():
        print(
            f"SUCCESS: {file_path.name} "
            f"({file_path.stat().st_size:,} bytes)"
        )
    else:
        print(f"MISSING: {file_path.name}")


Project directory: /content/careloop-no-show-guard
Data directory: /content/careloop-no-show-guard/data

Dataset verification:
SUCCESS: appointments.csv (84,008 bytes)
SUCCESS: clinics.csv (491 bytes)
SUCCESS: followup_notes.csv (2,336 bytes)


# **Part 1 — Python & Data Foundations**

### **Task 1.1 — CSV ingestion and validation**

In [4]:
print("Project directory:", PROJECT_DIR)
print("Appointments file:", APPOINTMENTS_PATH)
print("File exists:", APPOINTMENTS_PATH.exists())

Project directory: /content/careloop-no-show-guard
Appointments file: /content/careloop-no-show-guard/data/appointments.csv
File exists: True


### **Task 1.1 — Read the file the foundations way (5 marks)**

**Objective:**

Read appointments.csv using the built-in csv module,
convert numerical columns, and separate accepted
and rejected records.


* Using only the csv module (not pandas):
open appointments.csv with with open(..., newline=""), read it with
csv.DictReader, and print the header and row count.
* Write a function parse_row(row) that converts age, fee_inr and prior_no_shows to int and distance_km to float, returning the converted dictionary.
* Loop over every row; use try/except ValueError to put rows that fail into a rejected list and the rest into an accepted list. Save rejected with json.dump(..., indent=2) as rejected_rows.json.


In [5]:
# ************ TASK 1.1 : CSV Ingetion and data type validation ************

import csv
import json

# 1. Reading the oriinal appointment.csv file.

with open(
    APPOINTMENTS_PATH,
    mode="r",
    newline = "",
    encoding = "utf-8-sig"
) as file:

  reader = csv.DictReader(file)

  # Capture the CSV columns headings
  headers = reader.fieldnames

  # Read all records into a list
  raw_rows = list(reader)

# 2. Print the headers and total number of records.

print("CSV Headers:")
print(headers)

print("\n Total number of records:", len(raw_rows))

# Define the parse_row function (It parse the row to convert the string values to numerical columns)

def parse_row(row):

  # Make a copy so the oriinal row remains unchanged
  converted_row = row.copy()

  # Convert numerical columns
  # 'age', 'fee_inr', 'prior_no_shows', 'distance_km'

  converted_row["age"] = int(row["age"].strip())
  converted_row["fee_inr"] = int(row["fee_inr"].strip())
  converted_row["prior_no_shows"] = int(row["prior_no_shows"].strip())
  converted_row["distance_km"] = float(row["distance_km"].strip())

  return converted_row

# 4. Create accepted and rejected list.

accepted = []
rejected = []

# Track blank numerical values for verification

blank_counts = {
    "age": 0,
    "fee_inr": 0,
    "prior_no_shows": 0,
    "distance_km": 0
}

# 5. Process every row to convert the string columns to a specific numeric column and count no of blank records.

for row in raw_rows:
  try:
    converted_row  = parse_row(row)
    accepted.append(converted_row)

  except ValueError:
    rejected.append(row)

    # Identify blank numerical column in blank_counts

    for column in blank_counts:
      value = row.get(column)
      # print(f"{column}:{value}")
      if value is None or value.strip() == "" or value == 0:
        blank_counts[column] += 1

# 6. Save rejected records as JSON

rejected_path = PROJECT_DIR / "rejected_rows.json"

with open(rejected_path, mode="w", encoding="utf-8") as file:

    json.dump(
        rejected,
        file,
        indent=2,
        ensure_ascii=False
    )

# 7. Display processing results

print("\n--- TASK 1.1 RESULTS ---")

print("Total records:", len(raw_rows))
print("Accepted records:", len(accepted))
print("Rejected records:", len(rejected))

print("\nBlank numerical values in rejected records:")
for column, count in blank_counts.items():
    print(f"{column}: {count}")

# Check impossible ages that were still accepted
impossible_age_count = sum(
    1 for row in accepted
    if row["age"] < 1 or row["age"] > 100
)

print(
    "\nImpossible ages accepted during type conversion:",
    impossible_age_count
)

print("\nRejected JSON file saved:", rejected_path.exists())
print("Rejected JSON location:", rejected_path)

CSV Headers:
['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']

 Total number of records: 1020

--- TASK 1.1 RESULTS ---
Total records: 1020
Accepted records: 988
Rejected records: 32

Blank numerical values in rejected records:
age: 12
fee_inr: 0
prior_no_shows: 0
distance_km: 20

Impossible ages accepted during type conversion: 5

Rejected JSON file saved: True
Rejected JSON location: /content/careloop-no-show-guard/rejected_rows.json


## **Task 1.2 — Inspect and clean, pass 1**

### **Objectives**
1. Load the original appointments dataset using Pandas.
2. Inspect the shape, data types and missing values.
3. Identify exact duplicate records.
4. Remove duplicate records.
5. Standardise booking channel labels.
6. Convert date columns to datetime.
7. Remove records with missing appointment dates or status.

In [6]:
# *********** TASK 1.2 — DATA INSPECTION AND INITIAL CLEANING  ***********


# ============================================================
# TASK 1.2 — DATA INSPECTION AND INITIAL CLEANING
# ============================================================

# Step 1: Import Pandas

import pandas as pd

df = pd.read_csv(APPOINTMENTS_PATH)

print("STEP 1: INITIAL DATASET INSPECTION")
print("=" * 50)

# Step 3: Display dataset dimensions

print("\nDataset shape:")
print(df.shape)

# Step 4: Display column information

print("\nDataset information:")
df.info()

# Step 5: Check missing values

print("\nMissing values in each column:")
print(df.isnull().sum())

# Step 6: Check exact duplicate records

duplicate_count = df.duplicated().sum()

print("\nNumber of exact duplicate rows:")
print(duplicate_count)

# Step 7: Count distinct booking channel spellings
# before cleaning

print("\nUnique booking channels before cleaning:")
print(df["booking_channel"].nunique())

print(df["booking_channel"].unique())

STEP 1: INITIAL DATASET INSPECTION

Dataset shape:
(1020, 14)

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB

Missing

In [7]:
# ============================================================
# INITIAL CLEANING
# ============================================================

# Step 8: Remove exact duplicate rows

df = df.drop_duplicates().copy()

print("\nRows after removing duplicates:")
print(len(df))


Rows after removing duplicates:
1000


In [8]:
# Step 9: Standardise booking_channel

df["booking_channel"] = (
    df["booking_channel"]
    .str.strip()
    .str.title()
)

print("\nBooking channels after cleaning:")
print(df["booking_channel"].unique())

print(
    "Distinct booking channels after cleaning:",
    df["booking_channel"].nunique()
)


Booking channels after cleaning:
['Phone' 'Walk-In' 'App']
Distinct booking channels after cleaning: 3


In [9]:
# Step 11: Count missing dates and statuses

missing_appt_dates = df["appt_date"].isna().sum()
missing_status = df["status"].isna().sum()

print("\nMissing values after date conversion:")
print("Appointment dates:", missing_appt_dates)
print("Status:", missing_status)


Missing values after date conversion:
Appointment dates: 1
Status: 4


In [10]:
# Step 12: Remove rows with missing appointment date
# or missing status

rows_before_filter = len(df)

df = df.dropna(
    subset=["appt_date", "status"]
).copy()

rows_after_filter = len(df)

print("\nRows before filtering:", rows_before_filter)
print("Rows after filtering:", rows_after_filter)
print("Rows removed:", rows_before_filter - rows_after_filter)


Rows before filtering: 1000
Rows after filtering: 995
Rows removed: 5


In [11]:
# ============================================================
# TASK 1.2 SUMMARY
# ============================================================

print("\nTASK 1.2 COMPLETED")
print("=" * 50)

print("Final shape:", df.shape)
print("Remaining duplicate rows:", df.duplicated().sum())
print("Final booking channels:", df["booking_channel"].unique())


TASK 1.2 COMPLETED
Final shape: (995, 14)
Remaining duplicate rows: 0
Final booking channels: ['Phone' 'Walk-In' 'App']


In [12]:
# *********** TASK 1.2 — DATA INSPECTION AND INITIAL CLEANING  ***********


# ============================================================
# TASK 1.2 — DATA INSPECTION AND INITIAL CLEANING
# ============================================================

# Step 1: Import Pandas

import pandas as pd

# Step 2: Load the ORIGINAL appointments dataset
# Important: Start again from the source CSV, not accepted
# or rejected records from Task 1.1.

df = pd.read_csv(APPOINTMENTS_PATH)

print("STEP 1: INITIAL DATASET INSPECTION")
print("=" * 50)

# Step 3: Display dataset dimensions

print("\nDataset shape:")
print(df.shape)

# Step 4: Display column information

print("\nDataset information:")
df.info()

# Step 5: Check missing values

print("\nMissing values in each column:")
print(df.isnull().sum())

# Step 6: Check exact duplicate records

duplicate_count = df.duplicated().sum()

print("\nNumber of exact duplicate rows:")
print(duplicate_count)

# Step 7: Count distinct booking channel spellings
# before cleaning

print("\nUnique booking channels before cleaning:")
print(df["booking_channel"].nunique())

print(df["booking_channel"].unique())


# ============================================================
# INITIAL CLEANING
# ============================================================

# Step 8: Remove exact duplicate rows

df = df.drop_duplicates().copy()

print("\nRows after removing duplicates:")
print(len(df))


# Step 9: Standardise booking_channel

df["booking_channel"] = (
    df["booking_channel"]
    .str.strip()
    .str.title()
)

print("\nBooking channels after cleaning:")
print(df["booking_channel"].unique())

print(
    "Distinct booking channels after cleaning:",
    df["booking_channel"].nunique()
)


# Step 10: Convert date columns

# Unreadable appointment dates become NaT (missing datetime)
df["appt_date"] = pd.to_datetime(
    df["appt_date"],
    errors="coerce"
)

# Convert booked_on to datetime
df["booked_on"] = pd.to_datetime(df["booked_on"])


# Step 11: Count missing dates and statuses

missing_appt_dates = df["appt_date"].isna().sum()
missing_status = df["status"].isna().sum()

print("\nMissing values after date conversion:")
print("Appointment dates:", missing_appt_dates)
print("Status:", missing_status)


# Step 12: Remove rows with missing appointment date
# or missing status

rows_before_filter = len(df)

df = df.dropna(
    subset=["appt_date", "status"]
).copy()

rows_after_filter = len(df)

print("\nRows before filtering:", rows_before_filter)
print("Rows after filtering:", rows_after_filter)
print("Rows removed:", rows_before_filter - rows_after_filter)


# ============================================================
# TASK 1.2 SUMMARY
# ============================================================

print("\nTASK 1.2 COMPLETED")
print("=" * 50)

print("Final shape:", df.shape)
print("Remaining duplicate rows:", df.duplicated().sum())
print("Final booking channels:", df["booking_channel"].unique())



STEP 1: INITIAL DATASET INSPECTION

Dataset shape:
(1020, 14)

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB

Missing

# **Task 1.3: Handle missing values and outliers (5 marks)**

**This task introduces four important Data Science concepts:**
1. Invalid values.
2. Missing-value imputation.
3. Quartiles and Interquartile Range (IQR).
4. Outlier capping.

**Objectives:**
1. Identify invalid age values.
2. Replace invalid ages with missing values.
3. Fill missing age and distance using medians.
4. Calculate Q1, Q3 and IQR.
5. Calculate lower and upper outlier fences.
6. Cap distance and fee using NumPy.
7. Round distance and fee to the required precision.

In [13]:
# TASK 1.3 — MISSING VALUES AND OUTLIER TREATMENT
# ============================================================
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# STEP 1: Identify invalid age values
# ------------------------------------------------------------

# Invalid age: below 1 or above 100

invalid_age_mask = (
    (df["age"] < 1) |
    (df["age"] > 100)
)

#print(invalid_age_mask)

invalid_age_count = invalid_age_mask.sum()

print("Invalid age values:", invalid_age_count)

# Replace invalid ages with NaN
df.loc[invalid_age_mask, "age"] = np.nan

# ------------------------------------------------------------
# STEP 2: Calculate medians
# ------------------------------------------------------------

age_median = df["age"].median()
distance_median = df["distance_km"].median()

age_median = df["age"].median()
distance_median = df["distance_km"].median()

print("\nCalculated medians:")
print("Age median:", age_median)
print("Distance median:", distance_median)

# ------------------------------------------------------------
# STEP 3: Fill missing values using medians
# ------------------------------------------------------------

# Fill missing ages and round to the nearest integer
df["age"] = df["age"].fillna(round(age_median)).astype(int)

# Fill missing distances
df["distance_km"] = df["distance_km"].fillna(distance_median)

print("\nMissing values after imputation:")
print("Age:", df["age"].isna().sum())
print("Distance:", df["distance_km"].isna().sum())


# ------------------------------------------------------------
# STEP 4: Calculate quartiles and IQR for distance
# -----------------------------------------------------------

distance_q1 = df["distance_km"].quantile(0.25)
distance_q3 = df["distance_km"].quantile(0.75)

distance_iqr = distance_q3 - distance_q1

distance_lower_fence = distance_q1 - (1.5 * distance_iqr)
distance_upper_fence = distance_q3 + (1.5 * distance_iqr)

print("\nDISTANCE OUTLIER ANALYSIS")
print("Q1:", distance_q1)
print("Q3:", distance_q3)
print("IQR:", distance_iqr)
print("Lower fence:", distance_lower_fence)
print("Upper fence:", distance_upper_fence)


# ------------------------------------------------------------
# STEP 5: Calculate quartiles and IQR for fee
# ------------------------------------------------------------

fee_q1 = df["fee_inr"].quantile(0.25)
fee_q3 = df["fee_inr"].quantile(0.75)

fee_iqr = fee_q3 - fee_q1

fee_lower_fence = fee_q1 - (1.5 * fee_iqr)
fee_upper_fence = fee_q3 + (1.5 * fee_iqr)

print("\nFEE OUTLIER ANALYSIS")
print("Q1:", fee_q1)
print("Q3:", fee_q3)
print("IQR:", fee_iqr)
print("Lower fence:", fee_lower_fence)
print("Upper fence:", fee_upper_fence)

# ------------------------------------------------------------
# STEP 6: Cap distance outliers using NumPy
# ------------------------------------------------------------

# Preserve oriinal values before capping
distance_before_capping = df["distance_km"].copy()

# Apply lower and upper fences
df["distance_km"] = np.clip(
    df["distance_km"],
    distance_lower_fence,
    distance_upper_fence
)

# Count the values chaned by capping
distance_capped_count = (
    distance_before_capping != df["distance_km"]
).sum()

print("Capped distance count:",distance_capped_count)

# ------------------------------------------------------------
# STEP 7: Cap fee outliers using NumPy
# ------------------------------------------------------------

fee_before_capping = df["fee_inr"].copy()

df["fee_inr"] = np.clip(
    df["fee_inr"],
    fee_lower_fence,
    fee_upper_fence
)

fee_capped_count = (
    fee_before_capping != df["fee_inr"]
).sum()


# ------------------------------------------------------------
# STEP 8: Apply required rounding
# ------------------------------------------------------------
df["distance_km"] = df["distance_km"].round(1)

df["fee_inr"] = df["fee_inr"].round().astype(int)

# STEP 9: Display final results
# ------------------------------------------------------------

print("\n--- TASK 1.3 FINAL RESULTS ---")

print("Invalid ages identified:", invalid_age_count)
print("Age median:", age_median)
print("Distance median:", distance_median)

print("Distance values capped:", distance_capped_count)
print("Fee values capped:", fee_capped_count)

print("\nFinal age data type:", df["age"].dtype)
print("Final distance data type:", df["distance_km"].dtype)
print("Final fee data type:", df["fee_inr"].dtype)

print("\nMaximum distance after capping:",
      df["distance_km"].max())

print("Maximum fee after capping:",
      df["fee_inr"].max())

print("\nMissing values in age:",
      df["age"].isna().sum())

print("Missing values in distance:",
      df["distance_km"].isna().sum())

print("\nFinal dataset shape:", df.shape)

Invalid age values: 5

Calculated medians:
Age median: 48.0
Distance median: 8.25

Missing values after imputation:
Age: 0
Distance: 0

DISTANCE OUTLIER ANALYSIS
Q1: 4.625
Q3: 13.0
IQR: 8.375
Lower fence: -7.9375
Upper fence: 25.5625

FEE OUTLIER ANALYSIS
Q1: 500.0
Q3: 787.5
IQR: 287.5
Lower fence: 68.75
Upper fence: 1218.75
Capped distance count: 26

--- TASK 1.3 FINAL RESULTS ---
Invalid ages identified: 5
Age median: 48.0
Distance median: 8.25
Distance values capped: 26
Fee values capped: 3

Final age data type: int64
Final distance data type: float64
Final fee data type: int64

Maximum distance after capping: 25.6
Maximum fee after capping: 1219

Missing values in age: 0
Missing values in distance: 0

Final dataset shape: (990, 14)


In [14]:
print("DataFrame shape:", df.shape)
print("Duplicate rows:", df.duplicated().sum())
print("Missing appt_date:", df["appt_date"].isna().sum())
print("Missing status:", df["status"].isna().sum())

DataFrame shape: (990, 14)
Duplicate rows: 0
Missing appt_date: 0
Missing status: 0


# **Part 1 – Task 1.4: Feature Engineering and NumPy Analysis**


# **Step 1: Understand the concepts**
We'll create two new columns.

1. lead_days – Appointment booking lead time

This measures the number of days between booking an appointment and the actual appointment date.

2. no_show – Binary target variable

We'll convert appointment status into a numerical value:
- No-Show → 1
- Any other status → 0
This is called binary encoding.

# **Step 2: Create the two features**

lead_days and no_show


In [15]:

# Calculate the number of days between booking and appointment
df["lead_days"] = (
    df["appt_date"] - df["booked_on"]
).dt.days

# Convert appointment status into a binary variable
df["no_show"] = (
    df["status"].str.strip().str.lower() == "no-show"
).astype(int)

# Display a sample to verify the new columns
print("Sample of engineered features:")
display(
    df[[
        "appointment_id",
        "booked_on",
        "appt_date",
        "lead_days",
        "status",
        "no_show"
    ]].head(10)
)

print("\nLead days range:")
print("Minimum:", df["lead_days"].min())
print("Maximum:", df["lead_days"].max())

print("\nNo-show distribution:")
print(df["no_show"].value_counts().sort_index())




Sample of engineered features:


,appointment_id,booked_on,appt_date,lead_days,status,no_show
0,A0001,2025-01-24,2025-01-25,1,No-Show,1
1,A0002,2025-05-27,2025-06-06,10,Attended,0
2,A0003,2025-03-24,2025-03-30,6,Attended,0
4,A0004,2025-04-18,2025-04-22,4,Attended,0
5,A0005,2024-11-23,2025-01-02,40,No-Show,1
6,A0006,2025-04-17,2025-04-27,10,Attended,0
7,A0007,2025-02-10,2025-02-11,1,Attended,0
8,A0008,2025-01-12,2025-01-16,4,Attended,0
9,A0009,2025-03-10,2025-03-21,11,Attended,0
10,A0010,2025-03-10,2025-04-01,22,Attended,0



Lead days range:
Minimum: 1
Maximum: 45

No-show distribution:
no_show
0    772
1    218
Name: count, dtype: int64


# **Step 3: NumPy analysis**
Now we'll use NumPy to analyse four numerical features:
- Age
- Distance in kilometres
- Consultation fee
- Booking lead time
First, understand this important NumPy concept:

axis=0 calculates a statistic column-wise,
while axis=1 calculates it row-wise.

In [16]:
# Create a numpy array containing four numerical features

X = df[
  ["age", "distance_km", "fee_inr", "lead_days"]
].to_numpy()

# Calculate the mean of each array
feature_means = np.mean(X, axis=0)

print("Mean values:")
print("Age, Distance, Fee, Lead days")
print(np.round(feature_means, 2))

Mean values:
Age, Distance, Fee, Lead days
[ 48.17   9.5  635.71  11.82]


# **Step 4: Analyse longer booking lead times**
We want to know how many appointments were booked more than 14 days in advance.

In [20]:
# Identify appointments booked more than 14 days in advance.
long_lead_mask = df["lead_days"] > 14

#print(long_lead_mask)

# Count the appointments satisfying the condition whose lead_days > 14
long_lead_count = np.sum(long_lead_mask)

#print(long_lead_count)

# Calculate percentage
long_lead_percentage = (
  long_lead_count / len(df)
) * 100

print("Appointments with lead_days > 14:", long_lead_count)
print("Percentage:", round(long_lead_percentage, 1), "%")

Appointments with lead_days > 14: 296
Percentage: 29.9 %


Step 5: Detect unusual distances using Z-scores
A Z-score tells us how far a value is from the mean, measured in standard deviations.

The formula is:
~~~~python

z = x - mu / sigma (standard deviation)

~~~~

Where:
- \(x\) = individual distance
- \(\mu\) = mean distance
- \(\sigma\) = population standard deviation
We'll identify appointments where the absolute Z-score is greater than 2.

In [24]:
# Extract distance values as NumPy array
distances = df["distance_km"].to_numpy()

# Calculate the population mean and standard deviation
distance_mean = np.mean(distances)
distance_std = np.std(distances)

# Calculate Z-score for each appointment
distance_z_scores = (
    distances - distance_mean
) / distance_std

# Identify values with absolute Z-score greater than 2
unusual_distance_mask = np.abs(distance_z_scores) > 2

# Count unusual values
unusual_distance_count = np.sum(unusual_distance_mask)

print("Distance mean:", round(distance_mean, 2))
print("Distance standard deviation:", round(distance_std, 2))
print("Appointments with |Z-score| > 2:", unusual_distance_count)

Distance mean: 9.5
Distance standard deviation: 6.16
Appointments with |Z-score| > 2: 53


Step 6: Final verification

In [25]:

print("========== TASK 1.4 FINAL RESULTS ==========")

print("\nDataset shape:", df.shape)

print("\nLead days:")
print("Minimum:", df["lead_days"].min())
print("Maximum:", df["lead_days"].max())

print("\nNo-show analysis:")
print("Total no-shows:", df["no_show"].sum())
print(
    "No-show percentage:",
    round(df["no_show"].mean() * 100, 1),
    "%"
)

print("\nFeature means:")
print(np.round(feature_means, 2))

print("\nLong lead-time appointments:")
print(long_lead_count)
print(round(long_lead_percentage, 1), "%")

print("\nUnusual distances (|Z| > 2):")
print(unusual_distance_count)

print("\n============================================")


========== TASK 1.4 FINAL RESULTS ==========

Dataset shape: (990, 16)

Lead days:
Minimum: 1
Maximum: 45

No-show analysis:
Total no-shows: 218
No-show percentage: 22.0 %

Feature means:
[ 48.17   9.5  635.71  11.82]

Long lead-time appointments:
296
29.9 %

Unusual distances (|Z| > 2):
53



# **Part 1 – Task 1.5: Enrich Clinic Data Using the Open-Meteo API**

In [26]:

# Load the original clinic dataset
clinics = pd.read_csv(CLINICS_PATH)

# Display the dataset structure
print("Clinic dataset shape:", clinics.shape)

print("\nClinic column names:")
print(clinics.columns.tolist())

print("\nClinic dataset:")
display(clinics)

print("\nData types:")
print(clinics.dtypes)


Clinic dataset shape: (8, 6)

Clinic column names:
['clinic_id', 'clinic_name', 'city', 'tier', 'latitude', 'longitude']

Clinic dataset:


,clinic_id,clinic_name,city,tier,latitude,longitude
0,C01,Kothrud Family Clinic,Pune,Metro,18.5074,73.8077
1,C02,Hadapsar Care Centre,Pune,Metro,18.5089,73.9260
2,C03,Sitabuldi Health Point,Nagpur,Tier-2,21.1458,79.0882
3,C04,Malviya Nagar Clinic,Jaipur,Tier-2,26.8553,75.8150
4,C05,Vaishali Nagar Clinic,Jaipur,Tier-2,26.9124,75.7873
5,C06,Edappally Medical Centre,Kochi,Tier-2,10.0261,76.3085
6,C07,Vijay Nagar Clinic,Indore,Tier-2,22.7533,75.8937
7,C08,Gomti Nagar Care Point,Lucknow,Tier-2,26.8467,80.9462



Data types:
clinic_id       object
clinic_name     object
city            object
tier            object
latitude       float64
longitude      float64
dtype: object


# **Task 1.5 – API integration**

~~~~python
# API request parameters (sample data)
params = {
    "latitude": 18.5074,
    "longitude": 73.8077,
    "hourly": "temperature_2m",
    "forecast_days": 1
}
~~~~

In [28]:
import requests # requests handles HTTP communication
import time # time allows us to pause briefly before retrying a failed request.
import pandas as pd

# Implementaion of API Integration

# Open-Meteo API endpoint
API_URL = "https://api.open-meteo.com/v1/forecast"

# Store the temperature results for all clinics
temperatures = []

# Process each clinic one at a time
for _, clinic in clinics.iterrows():

    clinic_id = clinic["clinic_id"]
    clinic_name = clinic["clinic_name"]

    # Prepare API parameters using this clinic's coordinates
    params = {
        "latitude": clinic["latitude"],
        "longitude": clinic["longitude"],
        "hourly": "temperature_2m",
        "forecast_days": 1
    }

    temperature = None
    reason = None

    # First attempt + one retry
    for attempt in range(2):

        try:
            response = requests.get(
                API_URL,
                params=params,
                timeout=10
            )

            # Check HTTP response status
            if response.status_code != 200:
                reason = f"HTTP status {response.status_code}"
                break

            # Convert JSON response into a Python dictionary
            weather_data = response.json()

            # Extract the hourly temperature list
            hourly_data = weather_data.get("hourly", {})
            temperature_values = hourly_data.get(
                "temperature_2m"
            )

            # Check that temperature data is available
            if (
                not isinstance(temperature_values, list)
                or len(temperature_values) == 0
                or temperature_values[0] is None
            ):
                reason = "Hourly temperature data missing"
                break

            # Extract the first hourly temperature
            temperature = temperature_values[0]
            reason = None
            break

        except requests.exceptions.RequestException as error:

            reason = str(error)

            if attempt == 0:
                print(
                    f"Request failed for {clinic_id}. "
                    "Retrying once..."
                )
                time.sleep(1)

        except ValueError as error:

            reason = f"Invalid JSON response: {error}"
            break

    # Store the result, including None if unsuccessful
    temperatures.append(temperature)

    if temperature is not None:
        print(
            f"{clinic_id} - {clinic_name}: "
            f"{temperature} °C"
        )
    else:
        print(
            f"{clinic_id} - {clinic_name}: "
            f"Temperature unavailable. Reason: {reason}"
        )

# Add temperature results to the clinic DataFrame
clinics["temp_c"] = temperatures

print("\nAPI integration completed.")

C01 - Kothrud Family Clinic: 16.1 °C
C02 - Hadapsar Care Centre: 18.5 °C
C03 - Sitabuldi Health Point: 20.8 °C
C04 - Malviya Nagar Clinic: 18.9 °C
C05 - Vaishali Nagar Clinic: 18.9 °C
C06 - Edappally Medical Centre: 25.3 °C
C07 - Vijay Nagar Clinic: 19.3 °C
C08 - Gomti Nagar Care Point: 19.0 °C

API integration completed.


In [29]:

print("Enriched clinic dataset:")
display(clinics)

print("\nDataset shape:", clinics.shape)

print("\nMissing temperature values:")
print(clinics["temp_c"].isna().sum())

print("\nTemperature data type:")
print(clinics["temp_c"].dtype)

# Save the enriched clinic dataset
CLINICS_ENRICHED_PATH = (
    PROJECT_DIR / "clinics_enriched.csv"
)

clinics.to_csv(
    CLINICS_ENRICHED_PATH,
    index=False
)

print(
    "\nSaved successfully:",
    CLINICS_ENRICHED_PATH
)

Enriched clinic dataset:


,clinic_id,clinic_name,city,tier,latitude,longitude,temp_c
0,C01,Kothrud Family Clinic,Pune,Metro,18.5074,73.8077,16.1
1,C02,Hadapsar Care Centre,Pune,Metro,18.5089,73.9260,18.5
2,C03,Sitabuldi Health Point,Nagpur,Tier-2,21.1458,79.0882,20.8
3,C04,Malviya Nagar Clinic,Jaipur,Tier-2,26.8553,75.8150,18.9
4,C05,Vaishali Nagar Clinic,Jaipur,Tier-2,26.9124,75.7873,18.9
5,C06,Edappally Medical Centre,Kochi,Tier-2,10.0261,76.3085,25.3
6,C07,Vijay Nagar Clinic,Indore,Tier-2,22.7533,75.8937,19.3
7,C08,Gomti Nagar Care Point,Lucknow,Tier-2,26.8467,80.9462,19.0



Dataset shape: (8, 7)

Missing temperature values:
0

Temperature data type:
float64

Saved successfully: /content/careloop-no-show-guard/clinics_enriched.csv


# **Task 1.6 – Final dataset preparation and SQL export**

In [30]:

# Define the exact column order required by the assignment
final_columns = [
    "appointment_id",
    "patient_id",
    "clinic_id",
    "booked_on",
    "appt_date",
    "age",
    "gender",
    "distance_km",
    "fee_inr",
    "booking_channel",
    "visit_type",
    "reminder_sent",
    "prior_no_shows",
    "status",
    "lead_days",
    "no_show"
]

# Create a separate final DataFrame
appointments_final = df[final_columns].copy()

# Sort appointments by appointment_id
appointments_final = appointments_final.sort_values(
    by="appointment_id"
).reset_index(drop=True)

# Convert dates to YYYY-MM-DD format
appointments_final["booked_on"] = (
    pd.to_datetime(appointments_final["booked_on"])
    .dt.strftime("%Y-%m-%d")
)

appointments_final["appt_date"] = (
    pd.to_datetime(appointments_final["appt_date"])
    .dt.strftime("%Y-%m-%d")
)

# Display final dataset
print("Final appointment dataset:")
display(appointments_final.head())

print("\nDataset shape:", appointments_final.shape)
print("\nColumn order:")
print(appointments_final.columns.tolist())


Final appointment dataset:


,appointment_id,patient_id,clinic_id,booked_on,appt_date,age,gender,distance_km,fee_inr,booking_channel,visit_type,reminder_sent,prior_no_shows,status,lead_days,no_show
0,A0001,P509,C05,2025-01-24,2025-01-25,60,F,18.0,850,Phone,Consultation,Yes,1,No-Show,1,1
1,A0002,P046,C07,2025-05-27,2025-06-06,39,M,12.5,350,Walk-In,Follow-up,No,2,Attended,10,0
2,A0003,P009,C02,2025-03-24,2025-03-30,63,F,4.1,850,Walk-In,Consultation,No,0,Attended,6,0
3,A0004,P482,C01,2025-04-18,2025-04-22,52,F,7.5,750,App,Consultation,Yes,0,Attended,4,0
4,A0005,P251,C01,2024-11-23,2025-01-02,64,F,19.0,700,Phone,Consultation,No,0,No-Show,40,1



Dataset shape: (990, 16)

Column order:
['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status', 'lead_days', 'no_show']


In [31]:

# Expected final columns
expected_columns = final_columns

# Check column order
column_order_correct = (
    appointments_final.columns.tolist()
    == expected_columns
)

# Check row count
correct_row_count = len(appointments_final) == 990

# Check appointment ID uniqueness
unique_appointment_ids = (
    appointments_final["appointment_id"].is_unique
)

# Check missing values in all 16 columns
missing_values = appointments_final.isnull().sum()
total_missing_values = missing_values.sum()

# Check date formatting
booked_dates_valid = (
    appointments_final["booked_on"]
    .str.fullmatch(r"\d{4}-\d{2}-\d{2}")
    .all()
)

appt_dates_valid = (
    appointments_final["appt_date"]
    .str.fullmatch(r"\d{4}-\d{2}-\d{2}")
    .all()
)

# Print validation results
print("COLUMN ORDER CORRECT:", column_order_correct)
print("EXPECTED ROW COUNT:", correct_row_count)
print("UNIQUE APPOINTMENT IDs:", unique_appointment_ids)
print("TOTAL MISSING VALUES:", total_missing_values)
print("BOOKING DATE FORMAT VALID:", booked_dates_valid)
print("APPOINTMENT DATE FORMAT VALID:", appt_dates_valid)

print("\nMissing values by column:")
print(missing_values)

print("\nFinal dataset shape:", appointments_final.shape)


COLUMN ORDER CORRECT: True
EXPECTED ROW COUNT: True
UNIQUE APPOINTMENT IDs: True
TOTAL MISSING VALUES: 0
BOOKING DATE FORMAT VALID: True
APPOINTMENT DATE FORMAT VALID: True

Missing values by column:
appointment_id     0
patient_id         0
clinic_id          0
booked_on          0
appt_date          0
age                0
gender             0
distance_km        0
fee_inr            0
booking_channel    0
visit_type         0
reminder_sent      0
prior_no_shows     0
status             0
lead_days          0
no_show            0
dtype: int64

Final dataset shape: (990, 16)


In [32]:
APPOINTMENTS_CLEAN_PATH = (
    PROJECT_DIR / "appointments_clean.csv"
)

appointments_final.to_csv(
    APPOINTMENTS_CLEAN_PATH,
    index=False
)

print("Saved successfully:", APPOINTMENTS_CLEAN_PATH)

Saved successfully: /content/careloop-no-show-guard/appointments_clean.csv


In [33]:
# Generate the SQL file


import numpy as np
import pandas as pd

# Convert Python/Pandas values into SQL-compatible values
def sql_value(value):

    # Handle missing values
    if pd.isna(value):
        return "NULL"

    # Handle integers
    if isinstance(value, (int, np.integer)):
        return str(int(value))

    # Handle floating-point numbers
    if isinstance(value, (float, np.floating)):
        return format(float(value), ".10g")

    # Handle text and escape single quotes
    text_value = str(value).replace("'", "''")

    return f"'{text_value}'"


# Define the exact column order for the clinics table
clinic_columns = [
    "clinic_id",
    "clinic_name",
    "city",
    "tier",
    "latitude",
    "longitude",
    "temp_c"
]

# Define the exact column order for the appointments table
appointment_columns = [
    "appointment_id",
    "patient_id",
    "clinic_id",
    "booked_on",
    "appt_date",
    "age",
    "gender",
    "distance_km",
    "fee_inr",
    "booking_channel",
    "visit_type",
    "reminder_sent",
    "prior_no_shows",
    "status",
    "lead_days",
    "no_show"
]


# Store all SQL statements
sql_statements = []

sql_statements.append(
    "-- CareLoop Clinics: SQL data loading script"
)
sql_statements.append(
    "-- Clinics are inserted before appointments "
    "to satisfy the foreign key."
)
sql_statements.append("")


# Generate clinic INSERT statements FIRST
for _, row in clinics.iterrows():

    values = ", ".join(
        sql_value(row[column])
        for column in clinic_columns
    )

    statement = (
        "INSERT INTO clinics "
        f"({', '.join(clinic_columns)}) "
        f"VALUES ({values});"
    )

    sql_statements.append(statement)


sql_statements.append("")


# Generate appointment INSERT statements SECOND
for _, row in appointments_final.iterrows():

    values = ", ".join(
        sql_value(row[column])
        for column in appointment_columns
    )

    statement = (
        "INSERT INTO appointments "
        f"({', '.join(appointment_columns)}) "
        f"VALUES ({values});"
    )

    sql_statements.append(statement)


# Save the SQL file in the project root
LOAD_DATA_PATH = PROJECT_DIR / "load_data.sql"

with open(
    LOAD_DATA_PATH,
    "w",
    encoding="utf-8"
) as file:

    file.write("\n".join(sql_statements))
    file.write("\n")


print("SQL file generated successfully.")
print("File location:", LOAD_DATA_PATH)

print("\nSQL INSERT statement counts:")
print(
    "Clinic INSERT statements:",
    len(clinics)
)
print(
    "Appointment INSERT statements:",
    len(appointments_final)
)

print(
    "Total INSERT statements:",
    len(clinics) + len(appointments_final)
)


SQL file generated successfully.
File location: /content/careloop-no-show-guard/load_data.sql

SQL INSERT statement counts:
Clinic INSERT statements: 8
Appointment INSERT statements: 990
Total INSERT statements: 998


In [34]:
# Verify the SQL file

# Read the generated SQL file
with open(
    LOAD_DATA_PATH,
    "r",
    encoding="utf-8"
) as file:

    generated_sql = file.read()


# Count SQL INSERT statements
insert_count = generated_sql.count("INSERT INTO")

clinic_insert_count = generated_sql.count(
    "INSERT INTO clinics"
)

appointment_insert_count = generated_sql.count(
    "INSERT INTO appointments"
)

# Verify insertion order
first_clinic_position = generated_sql.find(
    "INSERT INTO clinics"
)

first_appointment_position = generated_sql.find(
    "INSERT INTO appointments"
)

print("========== SQL FILE VALIDATION ==========")

print("Clinic INSERT statements:", clinic_insert_count)
print("Appointment INSERT statements:", appointment_insert_count)
print("Total INSERT statements:", insert_count)

print(
    "Clinics inserted before appointments:",
    first_clinic_position < first_appointment_position
)

print(
    "Expected total INSERT statements:",
    insert_count == 998
)

print("\nFirst clinic SQL statement:")
print(
    generated_sql.splitlines()[
        next(
            i for i, line in enumerate(generated_sql.splitlines())
            if line.startswith("INSERT INTO clinics")
        )
    ]
)

print("\nFirst appointment SQL statement:")
print(
    next(
        line for line in generated_sql.splitlines()
        if line.startswith("INSERT INTO appointments")
    )
)

print("\n=========================================")


========== SQL FILE VALIDATION ==========
Clinic INSERT statements: 8
Appointment INSERT statements: 990
Total INSERT statements: 998
Clinics inserted before appointments: True
Expected total INSERT statements: True

First clinic SQL statement:
INSERT INTO clinics (clinic_id, clinic_name, city, tier, latitude, longitude, temp_c) VALUES ('C01', 'Kothrud Family Clinic', 'Pune', 'Metro', 18.5074, 73.8077, 16.1);

First appointment SQL statement:
INSERT INTO appointments (appointment_id, patient_id, clinic_id, booked_on, appt_date, age, gender, distance_km, fee_inr, booking_channel, visit_type, reminder_sent, prior_no_shows, status, lead_days, no_show) VALUES ('A0001', 'P509', 'C05', '2025-01-24', '2025-01-25', 60, 'F', 18, 850, 'Phone', 'Consultation', 'Yes', 1, 'No-Show', 1, 1);

